### Mitigation Embedding

In [10]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd

from huggingface_hub import InferenceClient
from pinecone import Pinecone

In [11]:
hf_token = os.getenv("HF_TOKEN")

client = InferenceClient(
    model="BAAI/bge-large-en-v1.5",
    token=hf_token
)

pc = Pinecone(api_key=os.getenv("pinecone_api_key"))

index_name = "mitigation-vector-db"

index = pc.Index(index_name)

In [12]:
json_folder = Path("../mitigation_data")

json_files = list(json_folder.glob("*.json"))

print("Total mitigation files:", len(json_files))

Total mitigation files: 31


In [13]:
def extract_documents(data):

    docs = []

    attack_name = data.get("attack_name")
    attack_type = data.get("attack_type")

    # ---------- MITRE MAPPING ----------
    mitre = data.get("mitre_attack_mapping", {})

    # ---- TACTICS ----
    tactics = mitre.get("tactic", [])

    if isinstance(tactics, dict):
        tactics = [tactics]

    for t in tactics:
        docs.append({
            "text": f"MITRE tactic {t.get('name')}: {t.get('description')}",
            "metadata": {
                "attack_name": attack_name,
                "attack_type": attack_type,
                "framework": "MITRE",
                "section": "tactic",
                "id": t.get("id"),
                "name": t.get("name")
            }
        })

    # ---- TECHNIQUES ----
    techniques = mitre.get("technique", [])

    if isinstance(techniques, dict):
        techniques = [techniques]

    for t in techniques:
        docs.append({
            "text": f"MITRE technique {t.get('name')}: {t.get('description')}",
            "metadata": {
                "attack_name": attack_name,
                "attack_type": attack_type,
                "framework": "MITRE",
                "section": "technique",
                "id": t.get("id"),
                "name": t.get("name")
            }
        })

    # ---------- MITRE MITIGATIONS ----------
    mitigations = data.get("mitre_mitigations", [])

    if isinstance(mitigations, dict):
        mitigations = [mitigations]

    for m in mitigations:
        docs.append({
            "text": f"Mitigation {m.get('name')}: {m.get('description')}",
            "metadata": {
                "attack_name": attack_name,
                "attack_type": attack_type,
                "framework": "MITRE",
                "section": "mitigation",
                "id": m.get("id"),
                "name": m.get("name")
            }
        })

    # ---------- NIST CONTROLS ----------
    nist = data.get("nist_recommendation", {})
    framework = nist.get("framework", "NIST")
    controls = nist.get("controls", [])

    if isinstance(controls, dict):
        controls = [controls]

    for c in controls:
        docs.append({
            "text": f"NIST control {c.get('name')}: {c.get('discussion')}",
            "metadata": {
                "attack_name": attack_name,
                "attack_type": attack_type,
                "framework": framework,
                "section": "control",
                "id": c.get("id"),
                "name": c.get("name")
            }
        })

    return docs

In [14]:
documents = []

for file in json_files:

    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)

    docs = extract_documents(data)

    documents.extend(docs)

print("Total documents:", len(documents))

Total documents: 166


In [15]:
vectors = []

for i, doc in enumerate(documents):

    emb = client.feature_extraction(doc["text"])

    emb = np.array(emb)
    emb = emb / np.linalg.norm(emb)

    vectors.append({
        "id": f"mitigation_{i}",
        "values": emb.tolist(),
        "metadata": doc["metadata"] | {
            "text": doc["text"][:500]
        }
    })

In [16]:
batch_size = 50

for i in range(0, len(vectors), batch_size):

    batch = vectors[i:i+batch_size]

    index.upsert(batch)

print("Mitigation knowledge uploaded")

Mitigation knowledge uploaded
